## 0. Settings

In [2]:
# Fix root
# Asuumes the current working directory is examples/experimental/
using Pkg
Pkg.activate(normpath(joinpath(pwd(), "..", "..")))
Pkg.instantiate()
Pkg.status()

  Activating project at `~/Projects/quantecon/ContinuousDPs.jl`


Project ContinuousDPs v0.1.0
Status `~/Projects/quantecon/ContinuousDPs.jl/Project.toml`
  [08854c51] BasisMatrices v0.8.0
  [6a86dc24] FiniteDiff v2.29.0
  [429524aa] Optim v2.0.1
  [fcd29c91] QuantEcon v0.17.1
  [37e2e46d] LinearAlgebra v1.12.0
  [9a3f8284] Random v1.11.0


In [3]:
using QuantEcon
using BasisMatrices
using ContinuousDPs
using Random
using PythonPlot
const plt = PythonPlot.pyplot;

## 1. Model

In [11]:
# Parameter Settings 
struct Params
    β::Float64
    λ::Float64
    A::Float64
    α::Float64
    δ::Float64
    ρ::Float64
    σϵ::Float64
end

p = Params(0.95, 1/3, 10.0, 0.34, 1.0, 0.90, 0.008)


Params(0.95, 0.3333333333333333, 10.0, 0.34, 1.0, 0.9, 0.008)

In [39]:
# Output
y(k, z, l) = z * p.A * k^p.α * (1 - l)^(1 - p.α)

# Calculate consumption and k prime based on Santos equation (7.4) ("unidimensional maximization")
function c_from_l(k, z, l)
    return z * p.A * k^p.α * (1 - l)^(-p.α) * (p.λ / (1 - p.λ)) * (1 - p.α) * l
end

function kprime_from_l(k, z, l)
    return y(k, z, l) + (1 - p.δ) * k - c_from_l(k, z, l)
end

# Reward and transition
function f(s, l) 
    k, logz = s
    z = exp(logz)
    if !(0 < l < 1)
        return -Inf
    end
    c = c_from_l(k, z, l)
    kp = kprime_from_l(k, z, l)
    if c <= 0 || kp < 0
        return -Inf
    end
    return p.λ*log(c) + (1 - p.λ)*log(l)
end

function g(s, l, e)
    k, logz = s
    z = exp(logz)
    kp = kprime_from_l(k, z, l)
    logzp = p.ρ*logz + e
    return (kp, logzp)
end

# Domains
logz_min, logz_max = -0.32, 0.32
k_min, k_max = 0.10, 10.0
x_lb(s), x_ub(s) = 1e-10, 1 - 1e-10

(1.0e-10, 0.9999999999)

In [40]:
# shocks, weights (Gauss-Hermite quadrature)
n_shocks = 7
shocks, weights = qnwnorm(n_shocks, 0.0, p.σϵ^2)

([-0.03000351774180594, -0.01893407528587633, -0.009235243157919746, 0.0, 0.009235243157919746, 0.01893407528587633, 0.03000351774180594], [0.0005482688559722181, 0.030757123967586456, 0.24012317860501223, 0.4571428571428571, 0.24012317860501223, 0.030757123967586456, 0.0005482688559722181])

In [41]:
# interp (Chebyshev polynomial)
nk, nlogz = 143, 9 # based on Santos (1999) Table 16
basis = Basis(
    ChebParams(nk, k_min, k_max),
    ChebParams(nlogz, logz_min, logz_max)
)

2 dimensional Basis on the hypercube formed by (0.1, -0.32) × (10.0, 0.32).
Basis families are Cheb × Cheb


In [42]:
# Build DP
cdp = ContinuousDP(f, g, p.β, shocks, weights, x_lb, x_ub, basis)

ContinuousDP{2, Vector{Float64}, Matrix{Float64}, typeof(f), typeof(g), typeof(x_lb), typeof(x_ub)}(Main.f, Main.g, 0.95, [-0.03000351774180594, -0.01893407528587633, -0.009235243157919746, 0.0, 0.009235243157919746, 0.01893407528587633, 0.03000351774180594], [0.0005482688559722181, 0.030757123967586456, 0.24012317860501223, 0.4571428571428571, 0.24012317860501223, 0.030757123967586456, 0.0005482688559722181], Main.x_lb, Main.x_ub, ContinuousDPs.Interp{2, Matrix{Float64}, Matrix{Float64}, LinearAlgebra.LU{Float64, Matrix{Float64}, Vector{Int64}}}(2 dimensional Basis on the hypercube formed by (0.1, -0.32) × (10.0, 0.32).
Basis families are Cheb × Cheb
, [0.10029863349399726 -0.31513848096390656; 0.10268748525162419 -0.31513848096390656; … ; 9.997312514748376 0.31513848096390656; 9.999701366506002 0.31513848096390656], ([0.10029863349399726, 0.10268748525162419, 0.10746403584626751, 0.11462597999313662, 0.1241698611558455, 0.1360910732146321, 0.15038386268938275, 0.1670413315164092, 0.1

In [ ]:
# Solve DP by VFI
res = solve(cdp, VFI; max_iter=500, tol = sqrt(eps()))